# Trial Inspector

Set one `generation_run_id` + `trial_id` below and re-run every cell (Run All) to see
everything about that trial in one place: the canonical scene/character data, the U/S/P
dialogue transcripts, the H1 judge prompts (with a check for whether they're really
condition-blind), and the full H1 metric breakdown per condition.

This is a drill-down tool for one trial at a time, separate from `results_dashboard.ipynb`
(which aggregates across trials/conditions).

In [130]:
import glob
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # this notebook lives in notebooks/, one level under the repo root
H1_PATH = REPO_ROOT / "output" / "evaluation" / "h1_consistency.jsonl"
TRAJECTORY_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "trajectory" / "*.jsonl")
ALL_ACTIONS_GLOB = str(REPO_ROOT / "output" / "record" / "*" / "all" / "*.jsonl")

CONDITION_ORDER = ["U", "S", "P"]


def load_jsonl(path):
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

In [131]:
def find_trajectory_records(generation_run_id, trial_id):
    """Search every trajectory file under output/record/*/trajectory/, not one hardcoded
    path, so this keeps working if more scenes/titles/models get added later."""
    matches = {}
    for path in sorted(glob.glob(TRAJECTORY_GLOB)):
        for record in load_jsonl(path):
            if record.get("generation_run_id") == generation_run_id and record.get("trial_id") == trial_id:
                matches[record["context_condition"]] = record
    return matches


def find_generation_calls(generation_run_id, trial_id, condition):
    """One line in the `all` record file per (trial, condition), saved once the whole
    trial finishes; its `record` list holds one {prompt, response, duration} dict per
    round, in round order (record[0] = round 1). Search every `all` file, not one
    hardcoded path, for the same reason as find_trajectory_records()."""
    for path in sorted(glob.glob(ALL_ACTIONS_GLOB)):
        for line in load_jsonl(path):
            if (
                line.get("generation_run_id") == generation_run_id
                and line.get("trial_id") == trial_id
                and line.get("context_condition") == condition
            ):
                return line["record"]
    return None


def find_h1_rows(generation_run_id, trial_id):
    by_condition = {condition: [] for condition in CONDITION_ORDER}
    for row in load_jsonl(H1_PATH):
        if row.get("generation_run_id") == generation_run_id and row.get("trial_id") == trial_id:
            by_condition[row["context_condition"]].append(row)
    for condition in by_condition:
        by_condition[condition].sort(key=lambda r: r["round"])
    return by_condition

## Set the trial to inspect

In [132]:
GENERATION_RUN_ID = "cb_hp_031_dumbledore_harry-1790133518139879000"
TRIAL_ID = "cb_hp_031_dumbledore_harry_t06"

trajectory_records = find_trajectory_records(GENERATION_RUN_ID, TRIAL_ID)
h1_rows = find_h1_rows(GENERATION_RUN_ID, TRIAL_ID)
generation_calls = {c: find_generation_calls(GENERATION_RUN_ID, TRIAL_ID, c) for c in CONDITION_ORDER}

missing_trajectory = [c for c in CONDITION_ORDER if c not in trajectory_records]
missing_h1 = [c for c in CONDITION_ORDER if not h1_rows[c]]
missing_generation = [c for c in CONDITION_ORDER if not generation_calls[c]]
if missing_trajectory:
    raise ValueError(f"No trajectory record found for conditions: {missing_trajectory}")
if missing_h1:
    raise ValueError(f"No H1 rows found for conditions: {missing_h1}")
if missing_generation:
    raise ValueError(f"No generation-call trace ('all' record) found for conditions: {missing_generation}")

print(f"Found trial {TRIAL_ID!r} in run {GENERATION_RUN_ID!r}: all of U/S/P present.")
print({c: len(h1_rows[c]) for c in CONDITION_ORDER}, "rounds scored per condition")

Found trial 'cb_hp_031_dumbledore_harry_t06' in run 'cb_hp_031_dumbledore_harry-1790133518139879000': all of U/S/P present.
{'U': 1, 'S': 1, 'P': 1} rounds scored per condition


### 1. Scene information

In [133]:
reference = trajectory_records["U"]
scene = reference["scene"]
player = reference["player"]
target = reference["target"]

print("[Scene]")
for key in ("event", "time", "location", "description"):
    print(f"{key.capitalize()}: {scene.get(key)}")

print()
print(f"[Player — {player['name']}]")
for key in ("position", "states", "description"):
    print(f"{key.capitalize()}: {player.get(key)}")

print()
print(f"[Target — {target['name']}]")
for key in ("position", "states", "description", "self_belief", "environment_belief"):
    print(f"{key.capitalize()}: {target.get(key)}")

print()
print("[Action Reference]")
print(reference["action_reference"])

# docs/research-contract.md §5: fields-and-values(U) = fields-and-values(S) = fields-and-values(P).
# Cheap, concrete check that this trial actually respects it, not just an assumption.
equivalent = all(
    (trajectory_records[c]["scene"], trajectory_records[c]["target"], trajectory_records[c]["action_reference"])
    == (scene, target, reference["action_reference"])
    for c in CONDITION_ORDER
)
print()
print(f"Scene/character/action_reference identical across U/S/P: {equivalent}")

[Scene]
Event: Harry Potter's Statement at the Magical Tribunal
Time: A certain summer night
Location: Inside the Magical Tribunal
Description: The atmosphere inside the Magical Tribunal is tense and solemn, with everyone present focusing on Harry Potter's statement.

[Player — Albus Dumbledore]
Position: Sitting in a chair next to Harry Potter, gaze firmly fixed on the judges and the jury.
States: Maintaining a calm and confident demeanor, defending Harry Potter.
Description: A wise old wizard with long white beard and hair, he is Harry Potter's headmaster and defender.

[Target — Harry Potter]
Position: Standing at the witness stand in the center of the tribunal, facing the judges and the jury.
States: Appearing nervous yet determined, he tries to recall and describe what happened that night.
Description: A young wizard with black hair and green eyes, he is the main focus of the trial.
Self_belief: 
Environment_belief: 

[Action Reference]
Harry Potter is describing to the tribunal h

In [134]:
def format_interaction(record):
    player_name = record["player"]["name"]
    target_name = record["target"]["name"]
    lines = []
    for turn in record["turns"]:
        lines.append(f"Round {turn['round']}")
        lines.append(f"{player_name}: {turn['player_dialogue']}")
        lines.append(f"{target_name}: {turn['npc_dialogue']}")
    return "\n".join(lines)


def print_generation_prompt_structure(condition):
    """Round 1's actual filled generation prompt for this condition -- this is where
    U/S/P genuinely differ in structure (unlike the H1 judge prompt, which is
    deliberately condition-blind). Later rounds follow the same structure with the
    rolling Observation/memory content updated each round; not reprinted per round
    here since the structure itself doesn't change, only that one field's content."""
    first_call = generation_calls[condition][0]
    print(first_call["prompt"])

### 2. U — interaction

**Prompt structure used to generate U's responses** (round 1, representative):

In [135]:
print_generation_prompt_structure("U")

Please act as Harry Potter and respond using the tone, manner and vocabulary Harry Potter would use. Use the following context to produce one appropriate line of dialogue.
Harry Potter's Statement at the Magical Tribunal
A certain summer night
Inside the Magical Tribunal
The atmosphere inside the Magical Tribunal is tense and solemn, with everyone present focusing on Harry Potter's statement.
Harry Potter
A young wizard with black hair and green eyes, he is the main focus of the trial.
Standing at the witness stand in the center of the tribunal, facing the judges and the jury.
Appearing nervous yet determined, he tries to recall and describe what happened that night.



Previous interaction:
Harry Potter:Harry Potter is describing to the tribunal his encounter with two Dementors and how he protected himself and his cousin.;Harry Potter:I heard a noise, and then everything went dark and cold. I tried twice, but only produced silver smoke. On the third try, I successfully conjured my Pat

In [136]:
print(format_interaction(trajectory_records["U"]))

Round 1
Albus Dumbledore: Strange — I could have sworn we were standing in the Ministry's atrium rather than inside the Tribunal chamber. Are we not?
Harry Potter: I know it might seem odd, but when those Dementors came after us, it felt like the whole world had turned upside down. I just wanted to protect my cousin, and I didn't know if I could do it, but I had to try.


### 3. S — interaction

**Prompt structure used to generate S's responses** (round 1, representative):

In [137]:
print_generation_prompt_structure("S")

Please act as Harry Potter  and respond using the tone, manner and vocabulary Harry Potter would use.
 The context and Harry Potter's information is as follows:
Event: Harry Potter's Statement at the Magical Tribunal
Time: A certain summer night
Location: Inside the Magical Tribunal
Description: The atmosphere inside the Magical Tribunal is tense and solemn, with everyone present focusing on Harry Potter's statement.
Character Profile:
Name: Harry Potter
Character Description: A young wizard with black hair and green eyes, he is the main focus of the trial.
Position: Standing at the witness stand in the center of the tribunal, facing the judges and the jury.
Current State: Appearing nervous yet determined, he tries to recall and describe what happened that night.
Self Belief: 
Environment Belief: 
Recent Memory(Ordered in reverse chronological order): 
Observation: Previous interaction:
Harry Potter:Harry Potter is describing to the tribunal his encounter with two Dementors and how he 

In [138]:
print(format_interaction(trajectory_records["S"]))

Round 1
Albus Dumbledore: Strange — I could have sworn we were standing in the Ministry's atrium rather than inside the Tribunal chamber. Are we not?
Harry Potter: Um, well, it does feel a bit like we’re in a different world right now, Professor, But I assure you, what happened that night was all too real. I was just trying to protect myself and my cousin, Dudley. I didn’t want to hurt anyone; I just wanted to get us out of there. It was terrifying, and I—I just acted on instinct. I know it’s hard to understand, but when those Dementors came, it felt like all the light had vanished. I had to fight back.


### 4. P — interaction

**Prompt structure used to generate P's responses** (round 1, representative):

In [139]:
print_generation_prompt_structure("P")

Please act as Harry Potter and respond using the tone, manner and vocabulary Harry Potter would use.
The context below is ordered from highest to lowest priority. Use all relevant context when it is compatible. If context conflicts, follow the lower priority number.
Priority 1 — current_world_constraints
Instruction: Must not be contradicted and overrides all lower priorities facts.
Time: A certain summer night
Location: Inside the Magical Tribunal
Position: Standing at the witness stand in the center of the tribunal, facing the judges and the jury.
Priority 2 — world_background
Instruction: Keep the response consistent with the fictional world.
Event: Harry Potter's Statement at the Magical Tribunal
Observation: Previous interaction:
Harry Potter:Harry Potter is describing to the tribunal his encounter with two Dementors and how he protected himself and his cousin.;Harry Potter:I heard a noise, and then everything went dark and cold. I tried twice, but only produced silver smoke. On t

In [140]:
print(format_interaction(trajectory_records["P"]))

Round 1
Albus Dumbledore: Strange — I could have sworn we were standing in the Ministry's atrium rather than inside the Tribunal chamber. Are we not?
Harry Potter: Um, yes, Professor Dumbledore, It did feel like we were in the Ministry, but it was, well, a lot more terrifying than that. I was just trying to protect myself and Dudley from those Dementors. They... they bring this awful cold, and it just felt like all hope was slipping away. I never meant to use magic like that in front of Muggles, but I had to do something. I just wanted to keep us safe.


### 5. Judge prompt (H1)

One prompt per round per condition. `[Scene]`/`[Character]`/`[Action Reference]` should be
identical across U/S/P (same values, per the equivalence check above); `[Prior turns]`/
`[Current turn]` are expected to differ, since that's the actual generated dialogue, which
does vary by condition. The check below only compares the part *before* `[Prior turns]`.

In [141]:
def prompt_prefix(prompt_text):
    marker = "[Prior turns]"
    index = prompt_text.find(marker)
    return prompt_text[:index] if index != -1 else prompt_text


rounds = [row["round"] for row in h1_rows["U"]]
for round_number in rounds:
    prompts = {
        condition: next(row["prompt"] for row in h1_rows[condition] if row["round"] == round_number)
        for condition in CONDITION_ORDER
    }
    prefixes = {condition: prompt_prefix(text) for condition, text in prompts.items()}
    identical = len(set(prefixes.values())) == 1

    print(f"===== Round {round_number} — scene/character/rubric identical across U/S/P: {identical} =====")
    for condition in CONDITION_ORDER:
        print(f"--- {condition} ---")
        print(prompts[condition])
        print()

===== Round 1 — scene/character/rubric identical across U/S/P: True =====
--- U ---
Please evaluate one turn of Harry Potter's dialogue against the supplied scene and character information, and against the character's own earlier turns in this conversation.

[Scene]
Event: Harry Potter's Statement at the Magical Tribunal
Time: A certain summer night
Location: Inside the Magical Tribunal
Description: The atmosphere inside the Magical Tribunal is tense and solemn, with everyone present focusing on Harry Potter's statement.

[Character — Harry Potter]
Position: Standing at the witness stand in the center of the tribunal, facing the judges and the jury.
Current State: Appearing nervous yet determined, he tries to recall and describe what happened that night.
Description: A young wizard with black hair and green eyes, he is the main focus of the trial.

[Action Reference]
Harry Potter is describing to the tribunal his encounter with two Dementors and how he protected himself and his cousin.

### 6. U — H1 metrics

In [142]:
def format_world_consistency(entry):
    """world_consistency has no top-level evidence field -- its decision is derived
    (in evaluate.py) from a claims list, so show that list rather than a single quote.
    "checked_against" is required on every claim now (shown as "conflicts with" for an
    inconsistent claim, "checked against" for a consistent one, including the literal
    "No supplied fact addresses this claim." placeholder when nothing applies). See the
    2026-09-16 provenance entries (claim-list redesign, checked_against rename, then
    required-on-every-claim).

    Each claim also carries "quote" (the verbatim response fragment it's drawn from)
    and "quote_verified" (whether that fragment actually appears in the NPC's real
    dialogue) -- a False here means the judge's "claim" may be misrepresenting what was
    actually said, worth checking by hand. See the 2026-09-23 provenance entry."""
    lines = [f"World Consistency: {entry['decision']}"]
    if not entry["claims"]:
        lines.append("  (no checkable claims)")
    for claim in entry["claims"]:
        verified_flag = "" if claim.get("quote_verified", True) else "  [UNVERIFIED QUOTE]"
        if claim["consistent"]:
            lines.append(f"  [consistent] {claim['claim']}{verified_flag}")
            lines.append(f"    checked against: {claim['checked_against']}")
        else:
            lines.append(f"  [CONTRADICTS] {claim['claim']}{verified_flag}")
            lines.append(f"    conflicts with: {claim['checked_against']}")
        lines.append(f"    quote: {claim.get('quote', '(none)')!r}")
    return "\n".join(lines)


def format_h1_metrics(rows):
    blocks = []
    for row in rows:
        decisions = row["decisions"]
        context_use = decisions["context_use"]
        no_contradiction = decisions["no_contradiction"]
        lines = [
            f"Round {row['round']}",
            f"Context Use: {context_use['decision']}",
            f"{context_use['evidence'] or '(none — N/A)'}",
            format_world_consistency(decisions["world_consistency"]),
            f"No Contradiction: {no_contradiction['decision']}",
            f"Contradicted round: {no_contradiction['contradicted_round'] if no_contradiction['contradicted_round'] is not None else '(none)'}",
            f"{no_contradiction['evidence'] or '(none — N/A)'}",
        ]
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


print(format_h1_metrics(h1_rows["U"]))

Round 1
Context Use: PASS
Elicited fact: Harry Potter is describing his encounter with Dementors. Response: 'when those Dementors came after us, it felt like the whole world had turned upside down.'
World Consistency: PASS
  [consistent] Dementors came after us
    checked against: Action Reference: Harry Potter is describing to the tribunal his encounter with two Dementors.
    quote: 'when those Dementors came after us'
  [consistent] it felt like the whole world had turned upside down
    checked against: No supplied fact addresses this claim.
    quote: 'it felt like the whole world had turned upside down'
  [consistent] I just wanted to protect my cousin
    checked against: Action Reference: how he protected himself and his cousin.
    quote: 'I just wanted to protect my cousin'
  [consistent] I didn't know if I could do it, but I had to try
    checked against: No supplied fact addresses this claim.
    quote: "I didn't know if I could do it, but I had to try"
No Contradiction: 

### 7. S — H1 metrics

In [143]:
print(format_h1_metrics(h1_rows["S"]))

Round 1
Context Use: PASS
Elicited fact: Harry Potter is describing his encounter with Dementors. Response: 'when those Dementors came, it felt like all the light had vanished. I had to fight back.'
World Consistency: PASS
  [consistent] it feels like we’re in a different world right now
    checked against: No supplied fact addresses this claim.
    quote: 'it does feel a bit like we’re in a different world right now'
  [consistent] what happened that night was all too real
    checked against: Action Reference: Harry Potter is describing to the tribunal his encounter with two Dementors and how he protected himself and his cousin.
    quote: 'what happened that night was all too real'
  [consistent] I was just trying to protect myself and my cousin, Dudley
    checked against: Action Reference: Harry Potter is describing to the tribunal his encounter with two Dementors and how he protected himself and his cousin.
    quote: 'I was just trying to protect myself and my cousin, Dudley'
 

### 8. P — H1 metrics

In [144]:
print(format_h1_metrics(h1_rows["P"]))

Round 1
Context Use: PASS
Elicited fact: Harry Potter is describing his encounter with Dementors. Response: 'I was just trying to protect myself and Dudley from those Dementors.'
World Consistency: PASS
  [consistent] it felt like we were in the Ministry
    checked against: No supplied fact addresses this claim.
    quote: 'It did feel like we were in the Ministry'
  [consistent] I was just trying to protect myself and Dudley from those Dementors
    checked against: Action Reference: Harry Potter is describing to the tribunal his encounter with two Dementors and how he protected himself and his cousin.
    quote: 'I was just trying to protect myself and Dudley from those Dementors'
  [consistent] Dementors bring this awful cold and it felt like all hope was slipping away
    checked against: No supplied fact addresses this claim.
    quote: 'They... they bring this awful cold, and it just felt like all hope was slipping away'
  [consistent] I never meant to use magic like that in fro